## Домашнее задание 5. Анализ данных на Spark SQL
Панкратов А.Д.


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [ ]:
pip install pyspark

In [ ]:
pip install findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
import pyspark
from pyspark.context import SparkContext, SparkConf
from pyspark.sql.session import SparkSession
spark = (
    SparkSession
    .builder
    .appName('Test_01')
    .config('spark.ui.port', '9311')
    .config('spark.executor.memoryOverhead', '1G')
    .config('spark.shuffle.service.enabled', 'true')
    .config('spark.dynamicAllocation.enabled', 'true')
    .config('spark.driver.extraClassPath', '/opt/spark/jars/sqljdbc42.jar')\
    .config('spark.executor.extraClassPath', '/opt/spark/jars/sqljdbc42.jar')\
    .getOrCreate()
)

In [ ]:
from pyspark.sql import functions as f

In [ ]:
spark

### Шаг 1. Создать таблицу, используя csv-файл.



In [ ]:
df = spark.read.option("header",True).option("sep",",").csv("Hotel.csv", inferSchema=True)

In [ ]:
df.show(5,0)

+--------+--------+----------+--------------+-----------+------------+-----------------+-----------+---------+----+-----+----+--------------+--------------+----------------------+------------------------------+--------------+----------------+------------+
|ID      |n_adults|n_children|weekend_nights|week_nights|meal_plan   |car_parking_space|room_type  |lead_time|year|month|date|market_segment|repeated_guest|previous_cancellations|previous_bookings_not_canceled|avg_room_price|special_requests|status      |
+--------+--------+----------+--------------+-----------+------------+-----------------+-----------+---------+----+-----+----+--------------+--------------+----------------------+------------------------------+--------------+----------------+------------+
|INN00001|2       |0         |1             |2          |Meal Plan 1 |0                |Room_Type 1|224      |2017|10   |2   |Offline       |0             |0                     |0                             |65.0          |0      

In [ ]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- n_adults: integer (nullable = true)
 |-- n_children: integer (nullable = true)
 |-- weekend_nights: integer (nullable = true)
 |-- week_nights: integer (nullable = true)
 |-- meal_plan: string (nullable = true)
 |-- car_parking_space: integer (nullable = true)
 |-- room_type: string (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- avg_room_price: double (nullable = true)
 |-- special_requests: integer (nullable = true)
 |-- status: string (nullable = true)



### Шаг 2. Создать (сгенерировать) таблицу calendar, который будет состоять из одного поля calendar_dt со всеми днями с 2017-01-01 по 2018-12-31.



In [ ]:
df_cal = spark.sql(f"""
    SELECT
        date_format(calendar_dt, 'yyyy-dd-MM') as calendar_dt
    FROM (
        SELECT
            explode(sequence(
                to_date('2017-01-01'),
                to_date('2018-12-31'),
                interval 1 day
            )) as calendar_dt
    )
""")

df_cal.show(5, 0)

+-----------+
|calendar_dt|
+-----------+
|2017-01-01 |
|2017-02-01 |
|2017-03-01 |
|2017-04-01 |
|2017-05-01 |
+-----------+
only showing top 5 rows


### Шаг 3. Выполнить следующие запросы:

#### 1. Вычислить среднее количество ночей, которые гости проводят в отеле (только для подтвержденных бронирований, с детализацией по месяцам и годам)

In [ ]:

from pyspark.sql.types import IntegerType

# Расчет общего количества ночей
df = df.withColumn(
    "total_nights",
    f.col("weekend_nights").cast(IntegerType()) + f.col("week_nights").cast(IntegerType())
)

# Фильтрация только подтвержденных бронирований
confirmed_bookings = df.filter(f.col("status") == "Not_Canceled")

# Группировка по году и месяцу с вычислением среднего
result = confirmed_bookings.groupBy("year", "month")\
    .agg(f.avg("total_nights").alias("avg_nights"))\
    .orderBy("year", "month")

result.show(10, 0)

+----+-----+------------------+
|year|month|avg_nights        |
+----+-----+------------------+
|2017|7    |3.0166666666666666|
|2017|8    |2.7189384800965017|
|2017|9    |2.6550783912747105|
|2017|10   |2.7032898820608318|
|2017|11   |2.7241935483870967|
|2017|12   |3.043046357615894 |
|2018|1    |2.7414141414141415|
|2018|2    |2.6891679748822606|
|2018|3    |3.0392038600723765|
|2018|4    |2.924755887421022 |
+----+-----+------------------+
only showing top 10 rows


#### 2. Определить ТОП-3 месяца по проценту отмененных броней за 2018 год.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

df_2018 = df.filter((f.col("year") == 2018)) # & (f.col("status") == "Canceled"))

cancellation_rate = df_2018.groupBy("month") \
    .agg(f.count("*").alias("total_bookings"),
        sum(when(f.col("status") == "Canceled", 1).otherwise(0)).alias("cancelled_bookings")) \
    .withColumn("cancellation_percentage",
      round((f.col("cancelled_bookings").cast("double") /
       f.col("total_bookings").cast("double")) * 100, 2)) \
    .orderBy(f.col("cancellation_percentage").desc())

cancellation_rate.select("month").show(3, 0)

+-----+
|month|
+-----+
|8    |
|10   |
|9    |
+-----+
only showing top 3 rows


#### 3. Вычислить среднее время на каждый месяц между бронированием и заездом в отель для подтвержденных броней.


In [ ]:
# Фильтрация подтвержденных бронирований
confirmed_bookings_2 = df.filter(
    (col("status") == "Not_Canceled") &
    (col("lead_time").isNotNull()))

# Группировка по месяцам и вычисление среднего lead_time
avg_lead_time = confirmed_bookings_2.groupBy("year", "month") \
    .agg(round(avg("lead_time"), 2).alias("avg_lead_time")) \
    .orderBy("year", "month")

avg_lead_time.show(10, 0)

+----+-----+-------------+
|year|month|avg_lead_time|
+----+-----+-------------+
|2017|7    |130.73       |
|2017|8    |35.08        |
|2017|9    |51.72        |
|2017|10   |55.89        |
|2017|11   |33.28        |
|2017|12   |46.75        |
|2018|1    |34.87        |
|2018|2    |30.53        |
|2018|3    |43.19        |
|2018|4    |62.49        |
+----+-----+-------------+
only showing top 10 rows


#### 4. Вычислить общую среднюю выручку на каждый месяц в каждом году, сгруппировав по всем типам бронирования для подтвержденных броней, и вывести это в виде сводной таблицы (PIVOT).


In [ ]:
from functools import reduce
from pyspark.sql import functions as F

# Фильтрация подтвержденных бронирований
confirmed_bookings = df.filter(col("status") == "Not_Canceled")

# Расчет общей выручки
revenue_df = confirmed_bookings \
  .withColumn("total_revenue",
    (col("weekend_nights") + col("week_nights")) * col("avg_room_price"))

# Группировка по месяцу, году и типу комнаты
grouped_df = revenue_df.groupBy("year", "month", "room_type") \
  .agg(sum("total_revenue").alias("total_revenue"))#.show(5, 0)

# Создание сводной таблицы
pivot_df = grouped_df.groupBy("year", "month") \
    .pivot("room_type") \
    .agg(sum("total_revenue")).fillna(0)#.show(10, 0)

# Создание списка колонок для суммирования (исключаем year и month)
cols_to_sum = [col for col in pivot_df.columns if col not in ("year", "month")]

# Суммируем колонки через reduce
sum_expression = reduce(lambda acc, col: acc + F.col(col), cols_to_sum, F.lit(0))

# Применение суммы в withColumn
final_df = pivot_df.withColumn("total_revenue", sum_expression)

# Сортировка результатов
result = final_df.orderBy("year", "month").show()

+----+-----+------------------+------------------+-----------+------------------+------------------+------------------+------------------+------------------+
|year|month|       Room_Type 1|       Room_Type 2|Room_Type 3|       Room_Type 4|       Room_Type 5|       Room_Type 6|       Room_Type 7|     total_revenue|
+----+-----+------------------+------------------+-----------+------------------+------------------+------------------+------------------+------------------+
|2017|    7|27878.479999999996|             196.4|        0.0|               0.0|               0.0|               0.0|               0.0|28074.879999999997|
|2017|    8|186667.08000000005|            1526.7|        0.0|           1546.75|               0.0|           14496.0|               0.0|204236.53000000006|
|2017|    9| 358495.3200000006|           1834.82|        0.0|          29036.75|             334.2|          14056.99|             639.0| 404397.0800000006|
|2017|   10|317950.90000000014|11084.140000000001|  

#### 5. Выявить ТОП-5 постоянных гостей, которые принесли наибольшую выручку за все время, и показать их долю в общей выручке от постоянных гостей. Использовать уникальный идентификатор брони как уникальный идентификатор гостя, предположив, что 1 бронь = 1 гость.


In [ ]:
# Фильтрация только подтвержденных бронирований постоянных гостей
regular_bookings = df.filter(
    (col("status") == "Not_Canceled") &
    (col("repeated_guest") == 1)
)

# Расчет выручки для каждой брони
revenue_df = regular_bookings.withColumn("total_revenue",
    (col("weekend_nights") + col("week_nights")) * col("avg_room_price")
)

# Группировка по ID и расчет общей выручки для каждого гостя
guest_revenue = revenue_df.groupBy("ID") \
    .agg(sum("total_revenue").alias("total_guest_revenue"))

# Расчет общей выручки всех постоянных гостей
total_regular_revenue = guest_revenue.select(sum("total_guest_revenue")).first()[0]

# Получение ТОП-5 гостей
top_5_guests = guest_revenue \
    .orderBy(col("total_guest_revenue").desc()) \
    .limit(5) \
    .withColumn("revenue_share",
        (col("total_guest_revenue") / total_regular_revenue) * 100
    )

# Форматирование результата
result = top_5_guests.select("ID", col("total_guest_revenue"),
    format_number("revenue_share", 2).alias("revenue_share_%"))

# Вывод результатов
result.show()

+--------+-------------------+---------------+
|      ID|total_guest_revenue|revenue_share_%|
+--------+-------------------+---------------+
|INN19235| 1754.3999999999999|           1.55|
|INN05222|              690.0|           0.61|
|INN14189|              665.0|           0.59|
|INN09923|              660.0|           0.58|
|INN25479|              650.0|           0.57|
+--------+-------------------+---------------+



#### 6. Вывести общее количество гостей на каждый день в отеле, отсортировав по убыванию дат, включая дни, когда отель пустует. Также рассчитать процент загрузки для каждого дня, если известно, что общая вместимость отеля 400 человек.

In [ ]:
df_cal.show(5)

+-----------+
|calendar_dt|
+-----------+
| 2017-01-01|
| 2017-02-01|
| 2017-03-01|
| 2017-04-01|
| 2017-05-01|
+-----------+
only showing top 5 rows


In [ ]:
# Фильтрация подтвержденных бронирований
confirmed_bookings = df.filter(f.col("status") == "Not_Canceled")

# Создание столбца с полной датой
confirmed_bookings = confirmed_bookings \
    .withColumn("full_date", f.concat_ws("-",
                       f.col("year"),
                       f.col("month"),
                       f.col("date"))) \
    .filter(F.col("full_date").isNotNull())  # Удаляем строки с ошибочными датами

# Расчет общего количества гостей
confirmed_bookings = confirmed_bookings \
    .withColumn("total_guests",
    f.col("n_adults").cast("int") + f.col("n_children").cast("int"))

# Группировка по дате и суммирование гостей
guests_per_day = confirmed_bookings \
    .groupBy("full_date") \
    .agg(f.sum("total_guests").alias("total_guests"))

# Объединение с полным календарём (df_cal должен содержать все даты)
full_calendar = df_cal.join(guests_per_day,
        f.col("calendar_dt") == f.col("full_date"), "left") \
    .fillna(0, subset=["total_guests"]) \
    .select(
        f.col("calendar_dt").alias("date"),
        f.col("total_guests").cast("double")  # Для корректного расчета процентов
    )

# Расчет загрузки и сортировка
result = full_calendar.withColumn("occupancy_percentage",
        (f.col("total_guests") / f.lit(400)) * f.lit(100)) \
    .orderBy(f.col("date").desc())  # Сортировка по убыванию дат

# Форматирование результата
final_result = result.select(
    f.col("date"),
    f.col("total_guests").cast("int").alias("total_guests"),
    f.format_number("occupancy_percentage", 2).alias("occupancy_percentage"))

# Вывод
final_result.show()

+----------+------------+--------------------+
|      date|total_guests|occupancy_percentage|
+----------+------------+--------------------+
|2018-31-12|           0|                0.00|
|2018-31-10|           0|                0.00|
|2018-31-08|           0|                0.00|
|2018-31-07|           0|                0.00|
|2018-31-05|           0|                0.00|
|2018-31-03|           0|                0.00|
|2018-31-01|           0|                0.00|
|2018-30-12|           0|                0.00|
|2018-30-11|           0|                0.00|
|2018-30-10|           0|                0.00|
|2018-30-09|           0|                0.00|
|2018-30-08|           0|                0.00|
|2018-30-07|           0|                0.00|
|2018-30-06|           0|                0.00|
|2018-30-05|           0|                0.00|
|2018-30-04|           0|                0.00|
|2018-30-03|           0|                0.00|
|2018-30-01|           0|                0.00|
|2018-29-12| 